# 📈 Options Pricer — Black-Scholes & Monte Carlo

> **Projet Finance Quantitative** — Valorisation d'options européennes par deux méthodes : formule analytique (Black-Scholes) et simulation stochastique (Monte Carlo).

---

## Sommaire
1. [Paramètres & Imports](#1)
2. [Modèle de Black-Scholes (analytique)](#2)
3. [Simulation Monte Carlo](#3)
4. [Comparaison des deux méthodes](#4)
5. [Graphique de convergence Monte Carlo](#5)
6. [Volatilité implicite (Bonus)](#6)

<a id='1'></a>
## 1. Paramètres & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

# Style matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'grid.linestyle': '--',
    'font.family': 'monospace',
    'figure.dpi': 120
})

np.random.seed(42)

# ── Paramètres de marché ──────────────────────────────────────
S0    = 100.0   # Prix actuel du sous-jacent
K     = 105.0   # Prix d'exercice (strike)
T     = 1.0     # Maturité (en années)
r     = 0.05    # Taux sans risque (annuel)
sigma = 0.20    # Volatilité implicite
q     = 0.00    # Taux de dividende continu

print("╔══════════════════════════════════════╗")
print("║      PARAMÈTRES DE L'OPTION          ║")
print("╠══════════════════════════════════════╣")
print(f"║  Spot (S₀)       : {S0:>8.2f}          ║")
print(f"║  Strike (K)      : {K:>8.2f}          ║")
print(f"║  Maturité (T)    : {T:>8.2f} an(s)     ║")
print(f"║  Taux r          : {r*100:>7.2f}%          ║")
print(f"║  Volatilité σ    : {sigma*100:>7.2f}%          ║")
print(f"║  Dividende q     : {q*100:>7.2f}%          ║")
print("╚══════════════════════════════════════╝")

<a id='2'></a>
## 2. Modèle de Black-Scholes (Analytique)

### Rappel théorique

Le modèle de Black-Scholes suppose que le sous-jacent suit un **mouvement brownien géométrique** :

$$dS_t = (r - q)\, S_t\, dt + \sigma\, S_t\, dW_t$$

Le prix d'une option européenne est donné par :

$$\boxed{C = S_0 e^{-qT} N(d_1) - K e^{-rT} N(d_2)}$$
$$\boxed{P = K e^{-rT} N(-d_2) - S_0 e^{-qT} N(-d_1)}$$

avec :
$$d_1 = \frac{\ln(S_0/K) + (r - q + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

In [ ]:
def black_scholes(S, K, T, r, sigma, q=0.0, option_type='call'):
    """
    Prix d'une option européenne via la formule de Black-Scholes.
    
    Paramètres
    ----------
    S           : float — Prix spot du sous-jacent
    K           : float — Strike
    T           : float — Maturité en années
    r           : float — Taux sans risque
    sigma       : float — Volatilité
    q           : float — Taux de dividende (défaut 0)
    option_type : str   — 'call' ou 'put'
    
    Retourne
    --------
    price  : float — Prix de l'option
    greeks : dict  — Delta, Gamma, Vega, Theta, Rho
    """
    if T <= 0:
        # Option à maturité : valeur intrinsèque
        if option_type == 'call':
            return max(S - K, 0), {}
        else:
            return max(K - S, 0), {}

    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        delta = np.exp(-q * T) * norm.cdf(d1)
        rho   = K * T * np.exp(-r * T) * norm.cdf(d2) / 100
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)
        delta = -np.exp(-q * T) * norm.cdf(-d1)
        rho   = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    # Greeks communs
    gamma = np.exp(-q * T) * norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega  = S * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(T) / 100
    theta = (- (S * np.exp(-q * T) * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
             - r * K * np.exp(-r * T) * norm.cdf(d2 if option_type == 'call' else -d2)
             + q * S * np.exp(-q * T) * norm.cdf(d1 if option_type == 'call' else -d1)) / 365

    greeks = {
        'Delta': delta,
        'Gamma': gamma,
        'Vega' : vega,
        'Theta': theta,
        'Rho'  : rho,
        'd1'   : d1,
        'd2'   : d2
    }
    return price, greeks


# ── Calcul ────────────────────────────────────────────────────
bs_call, greeks_call = black_scholes(S0, K, T, r, sigma, q, 'call')
bs_put,  greeks_put  = black_scholes(S0, K, T, r, sigma, q, 'put')

print("\n┌─────────────────────────────────────────────────────┐")
print("│           RÉSULTATS BLACK-SCHOLES                    │")
print("├──────────────────────┬──────────────┬───────────────┤")
print("│ Métrique             │    CALL      │     PUT       │")
print("├──────────────────────┼──────────────┼───────────────┤")
print(f"│ Prix                 │  {bs_call:>10.4f}  │ {bs_put:>10.4f}    │")
print(f"│ Delta                │  {greeks_call['Delta']:>10.4f}  │ {greeks_put['Delta']:>10.4f}    │")
print(f"│ Gamma                │  {greeks_call['Gamma']:>10.4f}  │ {greeks_put['Gamma']:>10.4f}    │")
print(f"│ Vega  (×1%)          │  {greeks_call['Vega']:>10.4f}  │ {greeks_put['Vega']:>10.4f}    │")
print(f"│ Theta (par jour)     │  {greeks_call['Theta']:>10.4f}  │ {greeks_put['Theta']:>10.4f}    │")
print(f"│ Rho   (×1%)          │  {greeks_call['Rho']:>10.4f}  │ {greeks_put['Rho']:>10.4f}    │")
print("└──────────────────────┴──────────────┴───────────────┘")

# Vérification parité put-call : C - P = S*e^(-qT) - K*e^(-rT)
parity_lhs = bs_call - bs_put
parity_rhs = S0 * np.exp(-q * T) - K * np.exp(-r * T)
print(f"\n✅ Parité Put-Call  |  C - P = {parity_lhs:.6f}  |  S·e⁻ᵠᵀ - K·e⁻ʳᵀ = {parity_rhs:.6f}")

<a id='3'></a>
## 3. Simulation Monte Carlo

### Principe

On simule $N$ trajectoires du sous-jacent jusqu'à la maturité en utilisant la **solution exacte** du MBG :

$$S_T = S_0 \exp\!\left[\left(r - q - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\, Z\right], \quad Z \sim \mathcal{N}(0, 1)$$

Le prix est estimé par :
$$\hat{C} = e^{-rT} \cdot \mathbb{E}[\max(S_T - K,\, 0)]$$

**Variance Reduction** : on utilise la technique des **variables antithétiques** pour réduire l'erreur standard.

In [ ]:
def monte_carlo_pricer(S, K, T, r, sigma, q=0.0, option_type='call',
                       n_simulations=100_000, antithetic=True, seed=42):
    """
    Prix Monte Carlo d'une option européenne.
    
    Paramètres
    ----------
    n_simulations : int  — Nombre de trajectoires
    antithetic    : bool — Variables antithétiques (réduction de variance)
    
    Retourne
    --------
    price  : float — Prix estimé
    se     : float — Erreur standard à 95%
    ci     : tuple — Intervalle de confiance à 95%
    """
    rng = np.random.default_rng(seed)
    
    if antithetic:
        Z = rng.standard_normal(n_simulations // 2)
        Z = np.concatenate([Z, -Z])          # Variables antithétiques
    else:
        Z = rng.standard_normal(n_simulations)
    
    # Prix terminal du sous-jacent
    ST = S * np.exp((r - q - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    
    # Payoff actualisé
    if option_type == 'call':
        payoffs = np.maximum(ST - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)
    
    discount = np.exp(-r * T)
    discounted = discount * payoffs
    
    price = discounted.mean()
    se    = discounted.std() / np.sqrt(n_simulations)   # Erreur standard
    ci    = (price - 1.96 * se, price + 1.96 * se)      # IC 95%
    
    return price, se, ci


# ── Pricing ───────────────────────────────────────────────────
N = 200_000

mc_call, se_call, ci_call = monte_carlo_pricer(S0, K, T, r, sigma, q, 'call', N)
mc_put,  se_put,  ci_put  = monte_carlo_pricer(S0, K, T, r, sigma, q, 'put',  N)

print(f"\n┌──────────────────────────────────────────────────────────────┐")
print(f"│            RÉSULTATS MONTE CARLO  (N = {N:,})          │")
print(f"├────────────────────────┬─────────────────┬───────────────────┤")
print(f"│ Métrique               │      CALL        │       PUT         │")
print(f"├────────────────────────┼─────────────────┼───────────────────┤")
print(f"│ Prix MC                │  {mc_call:>12.4f}   │  {mc_put:>12.4f}     │")
print(f"│ Erreur standard        │  {se_call:>12.6f}   │  {se_put:>12.6f}     │")
print(f"│ IC 95% [low, high]     │  [{ci_call[0]:.4f}, {ci_call[1]:.4f}]  │  [{ci_put[0]:.4f}, {ci_put[1]:.4f}]  │")
print(f"└────────────────────────┴─────────────────┴───────────────────┘")

<a id='4'></a>
## 4. Comparaison des deux méthodes

In [ ]:
# ── Tableau de comparaison ────────────────────────────────────
results = pd.DataFrame({
    'Méthode'       : ['Black-Scholes', 'Monte Carlo', 'Écart absolu', 'Écart relatif (%)'],
    'Call'          : [bs_call, mc_call, abs(bs_call - mc_call), abs(bs_call - mc_call)/bs_call*100],
    'Put'           : [bs_put,  mc_put,  abs(bs_put  - mc_put),  abs(bs_put  - mc_put) /bs_put *100],
})
results = results.set_index('Méthode')
results = results.round(6)
display(results)

# ── Surface des prix en fonction de S et σ ───────────────────
S_range     = np.linspace(70, 140, 50)
sigma_range = np.linspace(0.05, 0.50, 50)
SS, SSIG    = np.meshgrid(S_range, sigma_range)

call_surface = np.array([
    [black_scholes(s, K, T, r, sig, q, 'call')[0] for s in S_range]
    for sig in sigma_range
])

fig = plt.figure(figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

# ── Panel gauche : Prix Call/Put en fonction de S ────────────
ax1 = fig.add_subplot(121)
S_vals = np.linspace(60, 150, 200)
calls  = [black_scholes(s, K, T, r, sigma, q, 'call')[0] for s in S_vals]
puts   = [black_scholes(s, K, T, r, sigma, q, 'put')[0]  for s in S_vals]

ax1.plot(S_vals, calls, color='#58a6ff', lw=2.5, label='Call BS')
ax1.plot(S_vals, puts,  color='#f78166', lw=2.5, label='Put BS')
ax1.axvline(S0, color='#3fb950', linestyle='--', alpha=0.7, label=f'S₀ = {S0}')
ax1.axvline(K,  color='#e3b341', linestyle=':',  alpha=0.7, label=f'K = {K}')
ax1.scatter([S0], [bs_call], color='#58a6ff', s=100, zorder=5)
ax1.scatter([S0], [bs_put],  color='#f78166', s=100, zorder=5)
ax1.set_xlabel('Prix du sous-jacent S', fontsize=11)
ax1.set_ylabel('Prix de l\'option', fontsize=11)
ax1.set_title('Prix Black-Scholes vs S', fontsize=13, color='white', pad=12)
ax1.legend(framealpha=0.3, facecolor='#21262d', edgecolor='#30363d')

# ── Panel droit : Surface 3D Call ────────────────────────────
ax2 = fig.add_subplot(122, projection='3d')
ax2.set_facecolor('#161b22')
surf = ax2.plot_surface(SS, SSIG * 100, call_surface,
                         cmap='plasma', alpha=0.9, linewidth=0)
ax2.set_xlabel('Spot S', labelpad=8)
ax2.set_ylabel('Volatilité σ (%)', labelpad=8)
ax2.set_zlabel('Prix Call', labelpad=8)
ax2.set_title('Surface de prix (Call)', fontsize=13, color='white', pad=10)
ax2.tick_params(colors='#8b949e')
ax2.xaxis.pane.fill = False
ax2.yaxis.pane.fill = False
ax2.zaxis.pane.fill = False
fig.colorbar(surf, ax=ax2, shrink=0.5, aspect=10, pad=0.1)

plt.suptitle('Black-Scholes — Analyse des prix', fontsize=15, color='#c9d1d9', y=1.01)
plt.tight_layout()
plt.savefig('bs_surface.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

<a id='5'></a>
## 5. Convergence de Monte Carlo

On observe comment l'estimateur MC converge vers la valeur analytique Black-Scholes lorsque le nombre de simulations augmente.

La loi des grands nombres garantit que :
$$\hat{C}_N \xrightarrow[N\to\infty]{} C_{BS}$$

L'erreur standard décroît en $\mathcal{O}(1/\sqrt{N})$.

In [ ]:
# ── Convergence MC ────────────────────────────────────────────
n_values  = np.unique(np.round(np.logspace(1, 6, 80)).astype(int))
mc_prices = []
mc_errors = []

for n in n_values:
    p, se, _ = monte_carlo_pricer(S0, K, T, r, sigma, q, 'call',
                                   n_simulations=n, antithetic=True, seed=42)
    mc_prices.append(p)
    mc_errors.append(1.96 * se)

mc_prices = np.array(mc_prices)
mc_errors = np.array(mc_errors)

# ── Plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

# Panel gauche — Convergence du prix
ax = axes[0]
ax.semilogx(n_values, mc_prices, color='#58a6ff', lw=1.5, label='Prix MC', alpha=0.8)
ax.fill_between(n_values,
                mc_prices - mc_errors,
                mc_prices + mc_errors,
                alpha=0.25, color='#58a6ff', label='IC 95%')
ax.axhline(bs_call, color='#f78166', lw=2, linestyle='--', label=f'Black-Scholes = {bs_call:.4f}')
ax.set_xlabel('Nombre de simulations N (log)', fontsize=11)
ax.set_ylabel('Prix estimé (Call)', fontsize=11)
ax.set_title('Convergence de l\'estimateur MC', fontsize=13, color='white', pad=12)
ax.legend(framealpha=0.3, facecolor='#21262d', edgecolor='#30363d')
ax.set_xlim(n_values[0], n_values[-1])

# Panel droit — Décroissance de l'erreur standard
ax2 = axes[1]
ax2.loglog(n_values, mc_errors, color='#e3b341', lw=2, label='Erreur std (IC 95%)')

# Ligne théorique O(1/sqrt(N))
c_theory = mc_errors[0] * np.sqrt(n_values[0])
ax2.loglog(n_values, c_theory / np.sqrt(n_values),
           color='#3fb950', lw=1.5, linestyle=':', label=r'$\mathcal{O}(1/\sqrt{N})$')

ax2.set_xlabel('Nombre de simulations N (log)', fontsize=11)
ax2.set_ylabel('Erreur standard (log)', fontsize=11)
ax2.set_title('Décroissance de l\'erreur standard', fontsize=13, color='white', pad=12)
ax2.legend(framealpha=0.3, facecolor='#21262d', edgecolor='#30363d')
ax2.set_xlim(n_values[0], n_values[-1])

plt.suptitle('Monte Carlo — Analyse de convergence', fontsize=15, color='#c9d1d9', y=1.01)
plt.tight_layout()
plt.savefig('mc_convergence.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f"\nAvec N = 1 000 000 simulations :")
p_1m, se_1m, ci_1m = monte_carlo_pricer(S0, K, T, r, sigma, q, 'call', 1_000_000)
print(f"  Prix MC     = {p_1m:.6f}")
print(f"  Prix BS     = {bs_call:.6f}")
print(f"  Erreur std  = {se_1m:.6f}")
print(f"  IC 95%      = [{ci_1m[0]:.6f}, {ci_1m[1]:.6f}]")

<a id='6'></a>
## 6. ⭐ Bonus — Volatilité Implicite

### Principe

La **volatilité implicite** est la valeur de $\sigma$ qu'il faut injecter dans Black-Scholes pour retrouver le **prix de marché** observé d'une option. C'est la résolution de l'équation inverse :

$$\sigma^* = BS^{-1}(C_{\text{marché}})$$

On utilise la méthode de **Brent** (dichotomie robuste) pour résoudre numériquement cette équation implicite.

### Smile de volatilité

En pratique, la volatilité implicite varie avec le strike — c'est le **smile de volatilité**, ce qui constitue une limite du modèle BS.

In [ ]:
def implied_volatility(market_price, S, K, T, r, q=0.0,
                       option_type='call', tol=1e-8):
    """
    Calcule la volatilité implicite par la méthode de Brent.
    
    Retourne sigma_impl ou NaN si aucune solution trouvée.
    """
    # Bornes : volatilité entre 0.1% et 500%
    sigma_low, sigma_high = 1e-4, 5.0

    def objective(sigma):
        price, _ = black_scholes(S, K, T, r, sigma, q, option_type)
        return price - market_price

    # Vérifications de bornes
    if objective(sigma_low) * objective(sigma_high) > 0:
        return np.nan

    try:
        sigma_impl = brentq(objective, sigma_low, sigma_high, xtol=tol)
        return sigma_impl
    except ValueError:
        return np.nan


# ── Test sur notre option ────────────────────────────────────
# On vérifie la cohérence : IV du prix BS doit retrouver sigma
iv_call = implied_volatility(bs_call, S0, K, T, r, q, 'call')
iv_put  = implied_volatility(bs_put,  S0, K, T, r, q, 'put')

print(f"Vérification de cohérence :")
print(f"  σ original        = {sigma:.4f} ({sigma*100:.2f}%)")
print(f"  IV retrouvée (Call) = {iv_call:.6f} ({iv_call*100:.4f}%)")
print(f"  IV retrouvée (Put)  = {iv_put:.6f} ({iv_put*100:.4f}%)")

# ── Smile de volatilité (données synthétiques) ───────────────
strikes = np.linspace(70, 140, 30)

# Simulons un "smile" réaliste : skew + smile (SVI-like)
log_moneyness = np.log(strikes / S0)
sigma_market  = 0.20 + 0.05 * log_moneyness**2 - 0.02 * log_moneyness

# Prix de marché simulés puis IV extraite
mkt_prices = [black_scholes(S0, k, T, r, sig, q, 'call')[0]
              for k, sig in zip(strikes, sigma_market)]
ivs = [implied_volatility(p, S0, k, T, r, q, 'call')
       for p, k in zip(mkt_prices, strikes)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

# ── Panel gauche : Smile de volatilité ───────────────────────
ax1 = axes[0]
moneyness = strikes / S0
ax1.plot(moneyness, np.array(ivs) * 100, 'o-', color='#58a6ff',
         lw=2.5, ms=5, label='Vol. Implicite')
ax1.axhline(sigma * 100, color='#f78166', linestyle='--', lw=1.5,
            label=f'σ flat = {sigma*100:.0f}%')
ax1.axvline(1.0, color='#3fb950', linestyle=':', alpha=0.7, label='ATM (S=K)')
ax1.fill_between(moneyness, np.array(ivs) * 100, sigma * 100,
                 where=np.array(ivs) > sigma, alpha=0.15, color='#58a6ff')
ax1.fill_between(moneyness, np.array(ivs) * 100, sigma * 100,
                 where=np.array(ivs) < sigma, alpha=0.15, color='#f78166')
ax1.set_xlabel('Moneyness (K / S₀)', fontsize=11)
ax1.set_ylabel('Volatilité implicite (%)', fontsize=11)
ax1.set_title('Smile de volatilité implicite', fontsize=13, color='white', pad=12)
ax1.legend(framealpha=0.3, facecolor='#21262d', edgecolor='#30363d')

# ── Panel droit : Structure par terme (term structure) ───────
maturities = np.linspace(0.05, 3.0, 50)
# Terme structure simulé : hump shape
sigma_term = 0.15 + 0.08 * np.exp(-maturities) + 0.02 * maturities * np.exp(-0.5 * maturities)

ax2 = axes[1]
ax2.plot(maturities * 12, sigma_term * 100, color='#e3b341', lw=2.5)
ax2.fill_between(maturities * 12, sigma_term * 100, alpha=0.15, color='#e3b341')
ax2.axvline(T * 12, color='#f78166', linestyle='--', lw=1.5, label=f'T actuel = {T*12:.0f}m')
ax2.set_xlabel('Maturité (mois)', fontsize=11)
ax2.set_ylabel('Volatilité implicite ATM (%)', fontsize=11)
ax2.set_title('Structure par terme de la volatilité', fontsize=13, color='white', pad=12)
ax2.legend(framealpha=0.3, facecolor='#21262d', edgecolor='#30363d')

plt.suptitle('Volatilité Implicite — Smile & Term Structure', fontsize=15, color='#c9d1d9', y=1.01)
plt.tight_layout()
plt.savefig('iv_smile.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 🏁 Récapitulatif

In [ ]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║                    RÉCAPITULATIF FINAL                      ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  CALL                                                        ║")
print(f"║    Black-Scholes   : {bs_call:>10.4f}                              ║")
print(f"║    Monte Carlo     : {mc_call:>10.4f}  (N={N:,})              ║")
print(f"║    Écart absolu    : {abs(bs_call-mc_call):>10.6f}                              ║")
print(f"║    Écart relatif   : {abs(bs_call-mc_call)/bs_call*100:>10.4f}%                             ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  PUT                                                         ║")
print(f"║    Black-Scholes   : {bs_put:>10.4f}                              ║")
print(f"║    Monte Carlo     : {mc_put:>10.4f}  (N={N:,})              ║")
print(f"║    Écart absolu    : {abs(bs_put-mc_put):>10.6f}                              ║")
print(f"║    Écart relatif   : {abs(bs_put-mc_put)/bs_put*100:>10.4f}%                             ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  VOL. IMPLICITE (Call BS) : {iv_call*100:>8.4f}%  ≈ σ={sigma*100:.2f}%         ║")
print("╚══════════════════════════════════════════════════════════════╝")

print("\n📋 Conclusions :")
print("  • Black-Scholes fournit une formule fermée exacte (sous les hypothèses du modèle).")
print("  • Monte Carlo converge vers BS mais nécessite ~100k+ simulations pour ε < 0.01.")
print("  • Les variables antithétiques réduisent significativement la variance de l'estimateur.")
print("  • La volatilité implicite révèle le smile/skew absent du modèle BS constant.")
print("  • BS reste la référence analytique ; MC est plus flexible (exotiques, multi-actifs).")